# Web-Based Library Management System

**Course:** SE307 Programming with Python · Fall 2026–27
**Topic code:** SE307-13 · **Working as:** Group (4 members)
**Agreed extra feature(s):** Simulated e-mail notifications for overdue books
**Repository:** https://github.com/nazlinuralbayrak/se307-project-library-management-system

| Name | Student no. | GitHub username |
|---|---|---|
| Nazlınur Albayrak | 230201006 | @nazlinuralbayrak |
| Mehmet Ali Kanevetci | 230201041 | @MehmetAlikkk |
| Emir Sirmen Ersoy | 230201025 | @Sirmen3410 |
| Yaman Balcı | 220201081 | @yamanbalci |

Save the notebook to the repository as `notebooks/project.ipynb`.

## 1. Problem

Managing library inventory, tracking member borrowings, and monitoring overdue books manually is error-prone and time-consuming for librarians. This project provides a web-based library management system for librarians and registered members, backed by an SQLite database and an object-oriented Python backend with a Flask web interface. Members can search the catalogue and borrow or return books under due-date rules, while librarians manage the catalogue, monitor overdue loans, and view a statistics dashboard. Additionally, the system generates simulated e-mail warnings for overdue books, demonstrating how core library workflows and notifications can be automated in a modular Python application.

## 2. Data

The project uses synthetic data stored in `data/library.db` (SQLite database, created by us and free to use), initialized from `data/sample_books.csv` and the database setup scripts in `src/database.py`. No real people's personal data, passwords, or health records are stored. Below, the database schema and synthetic seed records are initialized and inspected via `pandas`.

## 3. Setup



In [1]:
import os
import sys
import sqlite3
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

In [2]:
from src.database import init_db, seed_data, get_connection

init_db()
seed_data()

conn = get_connection()
df_books = pd.read_sql_query("SELECT * FROM books", conn)
conn.close()

df_books.head()

Database and tables initialized successfully.
Database seed completed successfully.


,id,title,author,isbn,total_copies,available_copies
0,1,Clean Code,Robert C. Martin,978-0132350884,4,4
1,2,Python Crash Course,Eric Matthes,978-1593279288,3,3
2,3,Introduction to Algorithms,Thomas H. Cormen,978-0262033848,2,2
3,4,Design Patterns,Erich Gamma,978-0201633610,3,3
4,5,The Pragmatic Programmer,Andrew Hunt,978-0135957059,5,5


## 4. Design

The project follows a modular structure matching the repository layout (`app.py`, `src/`, `templates/`, `static/`, `data/`, `tests/`):

- **Web Entry Point & Templates (`app.py`, `templates/`, `static/`):** Flask routes and HTML views (`login.html`, `register.html`, `books.html`, `dashboard.html`, `notifications.html`) for librarian and member interactions.
- **Database & Models (`src/database.py`, `src/models.py`):** SQLite connection management, table schemas (`members`, `books`, `loans`, `notifications`), synthetic data seeding, and OOP domain classes (`Book`, `Member`, `Loan`, `Notification`).
- **Authentication (`src/auth.py`):** User registration, password hashing, session management, and librarian/member role checks.
- **Catalogue, Loans & Analytics (`src/loan_service.py`):** Book catalogue search, borrow/return rules, and statistics dashboard data/charts.
- **Overdue Notifications (`src/notification_service.py`):** Overdue loan checks, simulated e-mail warning generation, and notification log filtering.

**Chosen Data Structures:**
- **SQLite3 Relational Database (`data/library.db`):** Stores persistent records with primary/foreign keys and uniqueness constraints (`email`, `isbn`).
- **OOP Classes (`Book`, `Member`, `Loan`, `Notification`):** Encapsulate domain rules such as copy availability (`is_available()`, `borrow_copy()`, `return_copy()`) and due-date checks (`is_overdue()`).
- **Dictionaries & `pandas` DataFrames:** Transfer row records cleanly between SQLite queries, Flask templates, and dashboard analytics.

## 5. Implementation

The cell below initializes the SQLite database tables and populates synthetic book and user records.

In [3]:
# Part 1: Database schema initialization and synthetic data population
init_db()
seed_data()
print("Database and synthetic records initialized.")

Database and tables initialized successfully.
Database seed completed successfully.
Database and synthetic records initialized.


The cell below demonstrates the OOP domain models from `src/models.py` for book copy tracking and overdue loan detection.

In [4]:
# Part 2: Domain models (Book, Member, Loan, Notification)
from datetime import date, timedelta
from src.models import Book, Member, Loan, Notification

sample_book = Book("Clean Code", "Robert C. Martin", "9780132350884", total_copies=2, available_copies=2, book_id=1)
sample_member = Member("Sample Student", "student@library.com", role="member", member_id=2)
sample_loan = Loan(
    member_id=sample_member.id,
    book_id=sample_book.id,
    loan_date=date.today() - timedelta(days=20),
    due_date=date.today() - timedelta(days=6),
    status="active",
    loan_id=1
)

print("Book available:", sample_book.is_available())
print("Loan overdue:", sample_loan.is_overdue())

Book available: True
Loan overdue: True


## 6. Demonstration

The cell below inspects seeded records in `data/library.db` and demonstrates error handling when attempting to borrow a book with zero available copies.

In [5]:
# Demonstration
conn = get_connection()
books_count = conn.execute("SELECT COUNT(*) AS cnt FROM books").fetchone()["cnt"]
members_count = conn.execute("SELECT COUNT(*) AS cnt FROM members").fetchone()["cnt"]
conn.close()

print(f"Books in DB: {books_count}, Members in DB: {members_count}")

single_copy_book = Book("Python Crash Course", "Eric Matthes", "9781593279288", total_copies=1, available_copies=1)
single_copy_book.borrow_copy()
try:
    single_copy_book.borrow_copy()
except ValueError as exc:
    print("Expected error on unavailable copy:", exc)

Books in DB: 10, Members in DB: 2
Expected error on unavailable copy: No available copies left for 'Python Crash Course'.


## 7. Testing

Unit tests are organized under `tests/` using `pytest`, alongside core domain assertions shown below:

| Test Case | Input / Scenario | Expected Outcome |
|---|---|---|
| Normal borrow & return | `Book` with 2 copies; `borrow_copy()` then `return_copy()` | `available_copies` updates `2 -> 1 -> 2` |
| Boundary return | `return_copy()` when `available_copies == total_copies` | Raises `ValueError` |
| Invalid borrow | `borrow_copy()` when `available_copies == 0` | Raises `ValueError` |
| Overdue detection | Active loan past `due_date` vs. returned loan | `True` for overdue active loan, `False` for returned loan |

In [6]:
b = Book("Test Book", "Author", "1112223334445", total_copies=2, available_copies=2)
b.borrow_copy()
assert b.available_copies == 1
b.return_copy()
assert b.available_copies == 2

try:
    b.return_copy()
    assert False, "Expected ValueError when returning extra copy"
except ValueError:
    pass

overdue_loan = Loan(1, 1, date.today() - timedelta(days=15), date.today() - timedelta(days=1), status="active")
returned_loan = Loan(1, 1, date.today() - timedelta(days=15), date.today() - timedelta(days=1), return_date=date.today(), status="returned")
assert overdue_loan.is_overdue() is True
assert returned_loan.is_overdue() is False

print("All tests passed")

All tests passed


## 8. Limitations and next steps

**Limitations:**
- E-mail notifications for overdue books are simulated and logged in the database rather than dispatched via a live SMTP server.
- The system models a single library branch and does not include online fine payment processing.
- SQLite is well suited for local development and coursework, whereas production multi-user traffic would benefit from a client-server database.

**Next Steps:**
- Integrate a real SMTP or transactional e-mail service for automated overdue reminders.
- Expand `pytest` coverage across all Flask routes and edge cases.
- Add ISBN barcode lookup and reservation queue priority notifications.

## 9. Contributions

| Member | GitHub username | Responsibilities | Notebook sections / files | Approx. share of work |
|---|---|---|---|---|
| Nazlınur Albayrak | @nazlinuralbayrak | Database schema, models (OOP), synthetic data population | §2, §3, §4, §5, §7, `src/models.py`, `src/database.py`, `data/sample_books.csv` | 25% |
| Mehmet Ali Kanevetci | @MehmetAlikkk | Authentication, user roles (librarian/member), session management | `src/auth.py`, `templates/login.html`, `templates/register.html` | 25% |
| Yaman Balcı | @yamanbalci | Book catalogue search, loan & return logic, statistics dashboard | `src/loan_service.py`, `templates/books.html`, `templates/dashboard.html` | 25% |
| Emir Sirmen Ersoy | @Sirmen3410 | Overdue check logic, simulated e-mail notifications, log filtering | `src/notification_service.py`, `templates/notifications.html` | 25% |

**What each member did, in their own words (2–4 sentences each):**

- **Nazlınur Albayrak:** I designed and initialized the SQLite database schema in `src/database.py`, implemented the OOP domain classes (`Book`, `Member`, `Loan`, `Notification`) in `src/models.py`, and created the synthetic dataset and seeding workflow in `data/sample_books.csv`. I also structured the project repository and notebook documentation.
- **Mehmet Ali Kanevetci:** I implemented user authentication, password hashing, session handling, and librarian/member role access in `src/auth.py`, along with `templates/login.html` and `templates/register.html`.
- **Yaman Balcı:** I built the book catalogue search, borrow/return workflows, and statistics dashboard charts in `src/loan_service.py`, `templates/books.html`, and `templates/dashboard.html`.
- **Emir Sirmen Ersoy:** I developed the overdue loan check logic, simulated e-mail warning generation, and notification log filtering in `src/notification_service.py` and `templates/notifications.html`.

**Interim (Week 10) → Final:** Work remained divided across the four core modules (`database/models`, `auth`, `loan_service/dashboard`, and `notification_service`) and was merged incrementally into `main` via feature branches and pull requests.

## 10. External code, libraries and AI tools

| What | Used for | Where |
|---|---|---|
| Flask | Web interface and routing | `app.py`, `templates/` |
| SQLite3 | Relational database storage | `src/database.py`, §3, §6 |
| pandas, Matplotlib | Statistics, tabular inspection, and dashboard charts | `src/loan_service.py`, `templates/dashboard.html`, §3 |
| pytest | Unit testing core rules | `tests/` |
| AI assistant (Gemini) | Project guidance, architectural setup, Git troubleshooting, drafting | Project structure, notebook drafting |

## 11. References

- Flask documentation: https://flask.palletsprojects.com/
- Python `sqlite3` documentation: https://docs.python.org/3/library/sqlite3.html
- `pandas` documentation: https://pandas.pydata.org/docs/
- `matplotlib` documentation: https://matplotlib.org/stable/
- `pytest` documentation: https://docs.pytest.org/

---
Before submitting: Restart and run all cells with no errors, verify all fields are filled, save as `notebooks/project.ipynb` in the repository, and upload the same version to STIX.